# Análise exploratória | ZVCSD5077_001 | vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao

| Propriedade | Valor |
|---|---|
| Tabela | ``dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`` |
| Ambiente | DEV |
| Objeto | View |
| Colunas do DESCRIBE | 83 |
| Tipos | date, decimal(13,3), decimal(15,3), double, int, string |
| Clustering declarado | Não informado no CSV |
| Centro | `cod_centro` |
| Chave candidata | `num_documento` + `num_material` (validar na seção 4) |

**Como usar:** importar no Databricks e executar Run All. Células SQL independentes, somente leitura. Resultados são medições, não conclusões prévias. Para views, conferir no DESCRIBE EXTENDED se filtros da tela SAP são aplicados. Não executar cenários antes de analisar os resultados. Fontes: `DESCRIBE_ZVCSD5077_001.csv` e `01_PADROES_SQL_NOTEBOOKS.md`.

### Alertas do schema
| Severidade | Alerta |
|---|---|
| 🔴 ALTA | 14 campos tratados como potencialmente pessoais; amostras só com hash ou omitidos. |
| 🟡 MÉDIA | View: conferir comentário no DESCRIBE EXTENDED; DETAIL/HISTORY não executados. |
| 🟡 MÉDIA | Valores DOUBLE: conferir precisão e tolerância de 0,005 na comparação. |

### Índice
1 Metadados, 2 Volumetria, 3 Distribuição por centro, 4 Granularidade, 5 Duplicidade, 6 Preenchimento de todas as colunas, 6.1 Colunas sem fonte, 7 Cardinalidade, 8 Domínio, 9 Booleanas e indicadores, 10 Perfil numérico, 11 Totais para conciliação, 12 Datas, 13 Códigos, 14 Checksum, 15 Amostra, 16 Distribuição interna, 17 Freshness, 18 Específicas da transação, 19 Amostra ampliada, 20 Resumo. Inclui subseções 3.1 e 6.1.


## 1. Metadados

Inspeciona propriedades e comentário do objeto para conferir escopo e eventuais filtros da transação.


In [0]:
DESCRIBE EXTENDED `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`;


### 1.1. DETAIL/HISTORY da view

Não executados: estes comandos não são válidos para view. Consultar o comentário da view na seção 1.


## 2. Volumetria

Conta registros e entidades existentes, sem supor que campos de identificação formam uma chave.


In [0]:
SELECT COUNT(*) AS linhas, COUNT(DISTINCT `num_documento`) AS distintos_num_documento, COUNT(DISTINCT `num_material`) AS distintos_num_material, COUNT(DISTINCT `cod_centro`) AS centros_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`;


## 3. Distribuição por centro

Mede volumes e sinaliza centros entre 10 mil e 300 mil registros, sem criar cenários.


In [0]:
SELECT `cod_centro` AS centro, COUNT(*) AS linhas, CASE WHEN COUNT(*) BETWEEN 10000 AND 300000 THEN 'CANDIDATO A CENARIO' ELSE 'AVALIAR VOLUME' END AS avaliacao FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `cod_centro` ORDER BY linhas DESC;


## 3.1. Centros 4128 e 4014

Confirma a presença e o formato literal dos centros padrão; ainda não filtra os demais testes.


In [0]:
SELECT `cod_centro` AS centro_literal, LENGTH(TRIM(CAST(`cod_centro` AS STRING))) AS tamanho, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` WHERE TRIM(CAST(`cod_centro` AS STRING)) RLIKE '^0*(4128|4014)$' GROUP BY `cod_centro`, LENGTH(TRIM(CAST(`cod_centro` AS STRING))) ORDER BY centro_literal;


## 4. Granularidade

Compara 3 a 5 chaves candidatas. Uma razão igual a 1 é indício, não prova isolada de chave de negócio.


In [0]:
WITH d AS (SELECT 'num_documento' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento` FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) k UNION ALL SELECT 'num_documento + num_material' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento`, `num_material` FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) k UNION ALL SELECT 'num_documento + num_material + num_nota_fiscal' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento`, `num_material`, `num_nota_fiscal` FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) k UNION ALL SELECT 'num_documento + num_material + num_remessa' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento`, `num_material`, `num_remessa` FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) k UNION ALL SELECT 'num_documento + num_material + num_processo' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento`, `num_material`, `num_processo` FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) k), v AS (SELECT COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) SELECT d.chave_candidata, v.linhas, d.chaves_distintas, ROUND(v.linhas / NULLIF(d.chaves_distintas, 0), 4) AS linhas_por_chave FROM d CROSS JOIN v ORDER BY linhas_por_chave, chave_candidata;


## 5. Duplicidade

Separa repetições de chave entre linhas idênticas e versões diferentes, com hash da linha inteira sem exibir dados pessoais.


In [0]:
WITH h AS (SELECT t.`num_documento`, t.`num_material`, md5(to_json(struct(t.`num_documento`, t.`cod_empresa`, t.`cod_local_negocio`, t.`cod_codigo_cliente`, t.`num_nota_fiscal`, t.`num_einvoice`, t.`cod_chave_acesso`, t.`dt_data_documento`, t.`tp_categoria_nota_fiscal`, t.`ind_estornada`, t.`num_material`, t.`cod_container`, t.`num_documento_transporte`, t.`nm_cliente`, t.`desc_endereco_chegada`, t.`ind_impressa`, t.`ind_manual`, t.`cod_fatura`, t.`cod_direcao`, t.`desc_material`, t.`qt_quantidade`, t.`vl_frete`, t.`qt_peso_bruto`, t.`qt_peso_liquido`, t.`qt_peso_liquido_nf`, t.`sg_unidade_peso`, t.`cod_roteiro`, t.`desc_roteiro`, t.`cod_centro`, t.`num_processo`, t.`cod_evento_processo`, t.`cod_portaria_entrada`, t.`cod_portaria_saida`, t.`dt_data_entrada_portaria`, t.`hr_hora_entrada_portaria`, t.`dt_data_chamado`, t.`hr_hora_chamado`, t.`dt_data_encaminhado`, t.`hr_hora_encaminhado`, t.`dt_data_faturamento`, t.`hr_hora_faturamento`, t.`dt_data_saida_portaria`, t.`hr_hora_saida_portaria`, t.`dt_data_retornado`, t.`hr_hora_retornado`, t.`dt_data_liberacao_2a_vez`, t.`hr_hora_liberacao_2a_vez`, t.`num_remessa`, t.`cod_recebedor`, t.`nm_recebedor`, t.`desc_endereco_entrega`, t.`cod_cep_entrega`, t.`desc_bairro_entrega`, t.`nm_cidade_entrega`, t.`cod_incoterms_1`, t.`cod_incoterms_2`, t.`dt_data_emissao_mercadoria`, t.`nm_cidade`, t.`cod_transportador`, t.`cod_placa_cavalo`, t.`cod_transportador_auxiliar`, t.`dt_data_entrada_auxiliar`, t.`nm_motorista`, t.`cod_cpf_motorista`, t.`sg_unidade_peso_transporte`, t.`nm_nome`, t.`cod_empresa_auxiliar`, t.`cod_centro_destino`, t.`cod_area_carregamento`, t.`cod_ponto_carregamento`, t.`dt_data`, t.`hr_hora`, t.`dt_data_inicio_carregamento`, t.`hr_hora_inicio_carregamento`, t.`dt_data_fim_carregamento`, t.`hr_hora_fim_carregamento`, t.`cod_usuario_responsavel`, t.`sg_unidade_medida_base`, t.`sg_moeda`, t.`dt_data_sistema`, t.`dateingest`, t.`yearingest`, t.`monthingest`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` t), d AS (SELECT `num_documento`, `num_material`, COUNT(*) AS linhas, COUNT(DISTINCT hash_linha) AS versoes FROM h GROUP BY `num_documento`, `num_material` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COALESCE(SUM(linhas),0) AS linhas_em_chaves_repetidas, COALESCE(SUM(linhas - versoes),0) AS repeticoes_identicas, COALESCE(SUM(CASE WHEN versoes > 1 THEN 1 ELSE 0 END),0) AS chaves_com_versoes_diferentes FROM d;


## 5.1. Atributos variáveis nas chaves repetidas

Mede variação de atributos não pessoais nas chaves candidatas repetidas; ausência de variação não prova identidade da linha completa.


In [0]:
WITH d AS (SELECT `num_documento`, `num_material`, COUNT(*) AS linhas, COUNT(DISTINCT CAST(`num_documento` AS STRING)) AS `dist_num_documento`, COUNT(DISTINCT CAST(`cod_empresa` AS STRING)) AS `dist_cod_empresa`, COUNT(DISTINCT CAST(`cod_local_negocio` AS STRING)) AS `dist_cod_local_negocio`, COUNT(DISTINCT CAST(`cod_codigo_cliente` AS STRING)) AS `dist_cod_codigo_cliente`, COUNT(DISTINCT CAST(`num_nota_fiscal` AS STRING)) AS `dist_num_nota_fiscal`, COUNT(DISTINCT CAST(`num_einvoice` AS STRING)) AS `dist_num_einvoice`, COUNT(DISTINCT CAST(`dt_data_documento` AS STRING)) AS `dist_dt_data_documento`, COUNT(DISTINCT CAST(`tp_categoria_nota_fiscal` AS STRING)) AS `dist_tp_categoria_nota_fiscal`, COUNT(DISTINCT CAST(`ind_estornada` AS STRING)) AS `dist_ind_estornada`, COUNT(DISTINCT CAST(`num_material` AS STRING)) AS `dist_num_material`, COUNT(DISTINCT CAST(`num_documento_transporte` AS STRING)) AS `dist_num_documento_transporte`, COUNT(DISTINCT CAST(`ind_impressa` AS STRING)) AS `dist_ind_impressa` FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `num_documento`, `num_material` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COUNT_IF(`dist_num_documento` > 1) AS `chaves_variam_num_documento`, COUNT_IF(`dist_cod_empresa` > 1) AS `chaves_variam_cod_empresa`, COUNT_IF(`dist_cod_local_negocio` > 1) AS `chaves_variam_cod_local_negocio`, COUNT_IF(`dist_cod_codigo_cliente` > 1) AS `chaves_variam_cod_codigo_cliente`, COUNT_IF(`dist_num_nota_fiscal` > 1) AS `chaves_variam_num_nota_fiscal`, COUNT_IF(`dist_num_einvoice` > 1) AS `chaves_variam_num_einvoice`, COUNT_IF(`dist_dt_data_documento` > 1) AS `chaves_variam_dt_data_documento`, COUNT_IF(`dist_tp_categoria_nota_fiscal` > 1) AS `chaves_variam_tp_categoria_nota_fiscal`, COUNT_IF(`dist_ind_estornada` > 1) AS `chaves_variam_ind_estornada`, COUNT_IF(`dist_num_material` > 1) AS `chaves_variam_num_material`, COUNT_IF(`dist_num_documento_transporte` > 1) AS `chaves_variam_num_documento_transporte`, COUNT_IF(`dist_ind_impressa` > 1) AS `chaves_variam_ind_impressa` FROM d;


## 6. Preenchimento de todas as colunas

Mede nulos, vazios textuais, zeros e úteis em TODAS as colunas. Zero em código pode ser valor legítimo; veredito não equivale a defeito.


In [0]:
WITH perf AS (SELECT stack(24, 'num_documento', 'string', c0_n, c0_v, c0_z, 'cod_empresa', 'string', c1_n, c1_v, c1_z, 'cod_local_negocio', 'string', c2_n, c2_v, c2_z, 'cod_codigo_cliente', 'string', c3_n, c3_v, c3_z, 'num_nota_fiscal', 'string', c4_n, c4_v, c4_z, 'num_einvoice', 'string', c5_n, c5_v, c5_z, 'cod_chave_acesso', 'string', c6_n, c6_v, c6_z, 'dt_data_documento', 'date', c7_n, c7_v, c7_z, 'tp_categoria_nota_fiscal', 'string', c8_n, c8_v, c8_z, 'ind_estornada', 'string', c9_n, c9_v, c9_z, 'num_material', 'string', c10_n, c10_v, c10_z, 'cod_container', 'string', c11_n, c11_v, c11_z, 'num_documento_transporte', 'string', c12_n, c12_v, c12_z, 'nm_cliente', 'string', c13_n, c13_v, c13_z, 'desc_endereco_chegada', 'string', c14_n, c14_v, c14_z, 'ind_impressa', 'string', c15_n, c15_v, c15_z, 'ind_manual', 'string', c16_n, c16_v, c16_z, 'cod_fatura', 'string', c17_n, c17_v, c17_z, 'cod_direcao', 'string', c18_n, c18_v, c18_z, 'desc_material', 'string', c19_n, c19_v, c19_z, 'qt_quantidade', 'decimal(13,3)', c20_n, c20_v, c20_z, 'vl_frete', 'double', c21_n, c21_v, c21_z, 'qt_peso_bruto', 'decimal(15,3)', c22_n, c22_v, c22_z, 'qt_peso_liquido', 'decimal(15,3)', c23_n, c23_v, c23_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`num_documento` IS NULL) AS c0_n, COUNT_IF(`num_documento` IS NOT NULL AND LOWER(TRIM(`num_documento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c0_v, COUNT_IF(TRIM(`num_documento`) RLIKE '^0+([.,]0+)?$') AS c0_z, COUNT_IF(`cod_empresa` IS NULL) AS c1_n, COUNT_IF(`cod_empresa` IS NOT NULL AND LOWER(TRIM(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c1_v, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+([.,]0+)?$') AS c1_z, COUNT_IF(`cod_local_negocio` IS NULL) AS c2_n, COUNT_IF(`cod_local_negocio` IS NOT NULL AND LOWER(TRIM(`cod_local_negocio`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c2_v, COUNT_IF(TRIM(`cod_local_negocio`) RLIKE '^0+([.,]0+)?$') AS c2_z, COUNT_IF(`cod_codigo_cliente` IS NULL) AS c3_n, COUNT_IF(`cod_codigo_cliente` IS NOT NULL AND LOWER(TRIM(`cod_codigo_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c3_v, COUNT_IF(TRIM(`cod_codigo_cliente`) RLIKE '^0+([.,]0+)?$') AS c3_z, COUNT_IF(`num_nota_fiscal` IS NULL) AS c4_n, COUNT_IF(`num_nota_fiscal` IS NOT NULL AND LOWER(TRIM(`num_nota_fiscal`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c4_v, COUNT_IF(TRIM(`num_nota_fiscal`) RLIKE '^0+([.,]0+)?$') AS c4_z, COUNT_IF(`num_einvoice` IS NULL) AS c5_n, COUNT_IF(`num_einvoice` IS NOT NULL AND LOWER(TRIM(`num_einvoice`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c5_v, COUNT_IF(TRIM(`num_einvoice`) RLIKE '^0+([.,]0+)?$') AS c5_z, COUNT_IF(`cod_chave_acesso` IS NULL) AS c6_n, COUNT_IF(`cod_chave_acesso` IS NOT NULL AND LOWER(TRIM(`cod_chave_acesso`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c6_v, COUNT_IF(TRIM(`cod_chave_acesso`) RLIKE '^0+([.,]0+)?$') AS c6_z, COUNT_IF(`dt_data_documento` IS NULL) AS c7_n, CAST(0 AS BIGINT) AS c7_v, CAST(0 AS BIGINT) AS c7_z, COUNT_IF(`tp_categoria_nota_fiscal` IS NULL) AS c8_n, COUNT_IF(`tp_categoria_nota_fiscal` IS NOT NULL AND LOWER(TRIM(`tp_categoria_nota_fiscal`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c8_v, COUNT_IF(TRIM(`tp_categoria_nota_fiscal`) RLIKE '^0+([.,]0+)?$') AS c8_z, COUNT_IF(`ind_estornada` IS NULL) AS c9_n, COUNT_IF(`ind_estornada` IS NOT NULL AND LOWER(TRIM(`ind_estornada`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c9_v, COUNT_IF(TRIM(`ind_estornada`) RLIKE '^0+([.,]0+)?$') AS c9_z, COUNT_IF(`num_material` IS NULL) AS c10_n, COUNT_IF(`num_material` IS NOT NULL AND LOWER(TRIM(`num_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c10_v, COUNT_IF(TRIM(`num_material`) RLIKE '^0+([.,]0+)?$') AS c10_z, COUNT_IF(`cod_container` IS NULL) AS c11_n, COUNT_IF(`cod_container` IS NOT NULL AND LOWER(TRIM(`cod_container`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c11_v, COUNT_IF(TRIM(`cod_container`) RLIKE '^0+([.,]0+)?$') AS c11_z, COUNT_IF(`num_documento_transporte` IS NULL) AS c12_n, COUNT_IF(`num_documento_transporte` IS NOT NULL AND LOWER(TRIM(`num_documento_transporte`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c12_v, COUNT_IF(TRIM(`num_documento_transporte`) RLIKE '^0+([.,]0+)?$') AS c12_z, COUNT_IF(`nm_cliente` IS NULL) AS c13_n, COUNT_IF(`nm_cliente` IS NOT NULL AND LOWER(TRIM(`nm_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c13_v, COUNT_IF(TRIM(`nm_cliente`) RLIKE '^0+([.,]0+)?$') AS c13_z, COUNT_IF(`desc_endereco_chegada` IS NULL) AS c14_n, COUNT_IF(`desc_endereco_chegada` IS NOT NULL AND LOWER(TRIM(`desc_endereco_chegada`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c14_v, COUNT_IF(TRIM(`desc_endereco_chegada`) RLIKE '^0+([.,]0+)?$') AS c14_z, COUNT_IF(`ind_impressa` IS NULL) AS c15_n, COUNT_IF(`ind_impressa` IS NOT NULL AND LOWER(TRIM(`ind_impressa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c15_v, COUNT_IF(TRIM(`ind_impressa`) RLIKE '^0+([.,]0+)?$') AS c15_z, COUNT_IF(`ind_manual` IS NULL) AS c16_n, COUNT_IF(`ind_manual` IS NOT NULL AND LOWER(TRIM(`ind_manual`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c16_v, COUNT_IF(TRIM(`ind_manual`) RLIKE '^0+([.,]0+)?$') AS c16_z, COUNT_IF(`cod_fatura` IS NULL) AS c17_n, COUNT_IF(`cod_fatura` IS NOT NULL AND LOWER(TRIM(`cod_fatura`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c17_v, COUNT_IF(TRIM(`cod_fatura`) RLIKE '^0+([.,]0+)?$') AS c17_z, COUNT_IF(`cod_direcao` IS NULL) AS c18_n, COUNT_IF(`cod_direcao` IS NOT NULL AND LOWER(TRIM(`cod_direcao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c18_v, COUNT_IF(TRIM(`cod_direcao`) RLIKE '^0+([.,]0+)?$') AS c18_z, COUNT_IF(`desc_material` IS NULL) AS c19_n, COUNT_IF(`desc_material` IS NOT NULL AND LOWER(TRIM(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c19_v, COUNT_IF(TRIM(`desc_material`) RLIKE '^0+([.,]0+)?$') AS c19_z, COUNT_IF(`qt_quantidade` IS NULL) AS c20_n, CAST(0 AS BIGINT) AS c20_v, COUNT_IF(`qt_quantidade` = 0) AS c20_z, COUNT_IF(`vl_frete` IS NULL) AS c21_n, CAST(0 AS BIGINT) AS c21_v, COUNT_IF(`vl_frete` = 0) AS c21_z, COUNT_IF(`qt_peso_bruto` IS NULL) AS c22_n, CAST(0 AS BIGINT) AS c22_v, COUNT_IF(`qt_peso_bruto` = 0) AS c22_z, COUNT_IF(`qt_peso_liquido` IS NULL) AS c23_n, CAST(0 AS BIGINT) AS c23_v, COUNT_IF(`qt_peso_liquido` = 0) AS c23_z FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) a UNION ALL SELECT stack(24, 'qt_peso_liquido_nf', 'decimal(15,3)', c24_n, c24_v, c24_z, 'sg_unidade_peso', 'string', c25_n, c25_v, c25_z, 'cod_roteiro', 'string', c26_n, c26_v, c26_z, 'desc_roteiro', 'string', c27_n, c27_v, c27_z, 'cod_centro', 'string', c28_n, c28_v, c28_z, 'num_processo', 'string', c29_n, c29_v, c29_z, 'cod_evento_processo', 'string', c30_n, c30_v, c30_z, 'cod_portaria_entrada', 'string', c31_n, c31_v, c31_z, 'cod_portaria_saida', 'string', c32_n, c32_v, c32_z, 'dt_data_entrada_portaria', 'date', c33_n, c33_v, c33_z, 'hr_hora_entrada_portaria', 'string', c34_n, c34_v, c34_z, 'dt_data_chamado', 'date', c35_n, c35_v, c35_z, 'hr_hora_chamado', 'string', c36_n, c36_v, c36_z, 'dt_data_encaminhado', 'date', c37_n, c37_v, c37_z, 'hr_hora_encaminhado', 'string', c38_n, c38_v, c38_z, 'dt_data_faturamento', 'date', c39_n, c39_v, c39_z, 'hr_hora_faturamento', 'string', c40_n, c40_v, c40_z, 'dt_data_saida_portaria', 'date', c41_n, c41_v, c41_z, 'hr_hora_saida_portaria', 'string', c42_n, c42_v, c42_z, 'dt_data_retornado', 'date', c43_n, c43_v, c43_z, 'hr_hora_retornado', 'string', c44_n, c44_v, c44_z, 'dt_data_liberacao_2a_vez', 'date', c45_n, c45_v, c45_z, 'hr_hora_liberacao_2a_vez', 'string', c46_n, c46_v, c46_z, 'num_remessa', 'string', c47_n, c47_v, c47_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`qt_peso_liquido_nf` IS NULL) AS c24_n, CAST(0 AS BIGINT) AS c24_v, COUNT_IF(`qt_peso_liquido_nf` = 0) AS c24_z, COUNT_IF(`sg_unidade_peso` IS NULL) AS c25_n, COUNT_IF(`sg_unidade_peso` IS NOT NULL AND LOWER(TRIM(`sg_unidade_peso`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c25_v, COUNT_IF(TRIM(`sg_unidade_peso`) RLIKE '^0+([.,]0+)?$') AS c25_z, COUNT_IF(`cod_roteiro` IS NULL) AS c26_n, COUNT_IF(`cod_roteiro` IS NOT NULL AND LOWER(TRIM(`cod_roteiro`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c26_v, COUNT_IF(TRIM(`cod_roteiro`) RLIKE '^0+([.,]0+)?$') AS c26_z, COUNT_IF(`desc_roteiro` IS NULL) AS c27_n, COUNT_IF(`desc_roteiro` IS NOT NULL AND LOWER(TRIM(`desc_roteiro`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c27_v, COUNT_IF(TRIM(`desc_roteiro`) RLIKE '^0+([.,]0+)?$') AS c27_z, COUNT_IF(`cod_centro` IS NULL) AS c28_n, COUNT_IF(`cod_centro` IS NOT NULL AND LOWER(TRIM(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c28_v, COUNT_IF(TRIM(`cod_centro`) RLIKE '^0+([.,]0+)?$') AS c28_z, COUNT_IF(`num_processo` IS NULL) AS c29_n, COUNT_IF(`num_processo` IS NOT NULL AND LOWER(TRIM(`num_processo`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c29_v, COUNT_IF(TRIM(`num_processo`) RLIKE '^0+([.,]0+)?$') AS c29_z, COUNT_IF(`cod_evento_processo` IS NULL) AS c30_n, COUNT_IF(`cod_evento_processo` IS NOT NULL AND LOWER(TRIM(`cod_evento_processo`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c30_v, COUNT_IF(TRIM(`cod_evento_processo`) RLIKE '^0+([.,]0+)?$') AS c30_z, COUNT_IF(`cod_portaria_entrada` IS NULL) AS c31_n, COUNT_IF(`cod_portaria_entrada` IS NOT NULL AND LOWER(TRIM(`cod_portaria_entrada`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c31_v, COUNT_IF(TRIM(`cod_portaria_entrada`) RLIKE '^0+([.,]0+)?$') AS c31_z, COUNT_IF(`cod_portaria_saida` IS NULL) AS c32_n, COUNT_IF(`cod_portaria_saida` IS NOT NULL AND LOWER(TRIM(`cod_portaria_saida`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c32_v, COUNT_IF(TRIM(`cod_portaria_saida`) RLIKE '^0+([.,]0+)?$') AS c32_z, COUNT_IF(`dt_data_entrada_portaria` IS NULL) AS c33_n, CAST(0 AS BIGINT) AS c33_v, CAST(0 AS BIGINT) AS c33_z, COUNT_IF(`hr_hora_entrada_portaria` IS NULL) AS c34_n, COUNT_IF(`hr_hora_entrada_portaria` IS NOT NULL AND LOWER(TRIM(`hr_hora_entrada_portaria`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c34_v, COUNT_IF(TRIM(`hr_hora_entrada_portaria`) RLIKE '^0+([.,]0+)?$') AS c34_z, COUNT_IF(`dt_data_chamado` IS NULL) AS c35_n, CAST(0 AS BIGINT) AS c35_v, CAST(0 AS BIGINT) AS c35_z, COUNT_IF(`hr_hora_chamado` IS NULL) AS c36_n, COUNT_IF(`hr_hora_chamado` IS NOT NULL AND LOWER(TRIM(`hr_hora_chamado`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c36_v, COUNT_IF(TRIM(`hr_hora_chamado`) RLIKE '^0+([.,]0+)?$') AS c36_z, COUNT_IF(`dt_data_encaminhado` IS NULL) AS c37_n, CAST(0 AS BIGINT) AS c37_v, CAST(0 AS BIGINT) AS c37_z, COUNT_IF(`hr_hora_encaminhado` IS NULL) AS c38_n, COUNT_IF(`hr_hora_encaminhado` IS NOT NULL AND LOWER(TRIM(`hr_hora_encaminhado`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c38_v, COUNT_IF(TRIM(`hr_hora_encaminhado`) RLIKE '^0+([.,]0+)?$') AS c38_z, COUNT_IF(`dt_data_faturamento` IS NULL) AS c39_n, CAST(0 AS BIGINT) AS c39_v, CAST(0 AS BIGINT) AS c39_z, COUNT_IF(`hr_hora_faturamento` IS NULL) AS c40_n, COUNT_IF(`hr_hora_faturamento` IS NOT NULL AND LOWER(TRIM(`hr_hora_faturamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c40_v, COUNT_IF(TRIM(`hr_hora_faturamento`) RLIKE '^0+([.,]0+)?$') AS c40_z, COUNT_IF(`dt_data_saida_portaria` IS NULL) AS c41_n, CAST(0 AS BIGINT) AS c41_v, CAST(0 AS BIGINT) AS c41_z, COUNT_IF(`hr_hora_saida_portaria` IS NULL) AS c42_n, COUNT_IF(`hr_hora_saida_portaria` IS NOT NULL AND LOWER(TRIM(`hr_hora_saida_portaria`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c42_v, COUNT_IF(TRIM(`hr_hora_saida_portaria`) RLIKE '^0+([.,]0+)?$') AS c42_z, COUNT_IF(`dt_data_retornado` IS NULL) AS c43_n, CAST(0 AS BIGINT) AS c43_v, CAST(0 AS BIGINT) AS c43_z, COUNT_IF(`hr_hora_retornado` IS NULL) AS c44_n, COUNT_IF(`hr_hora_retornado` IS NOT NULL AND LOWER(TRIM(`hr_hora_retornado`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c44_v, COUNT_IF(TRIM(`hr_hora_retornado`) RLIKE '^0+([.,]0+)?$') AS c44_z, COUNT_IF(`dt_data_liberacao_2a_vez` IS NULL) AS c45_n, CAST(0 AS BIGINT) AS c45_v, CAST(0 AS BIGINT) AS c45_z, COUNT_IF(`hr_hora_liberacao_2a_vez` IS NULL) AS c46_n, COUNT_IF(`hr_hora_liberacao_2a_vez` IS NOT NULL AND LOWER(TRIM(`hr_hora_liberacao_2a_vez`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c46_v, COUNT_IF(TRIM(`hr_hora_liberacao_2a_vez`) RLIKE '^0+([.,]0+)?$') AS c46_z, COUNT_IF(`num_remessa` IS NULL) AS c47_n, COUNT_IF(`num_remessa` IS NOT NULL AND LOWER(TRIM(`num_remessa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c47_v, COUNT_IF(TRIM(`num_remessa`) RLIKE '^0+([.,]0+)?$') AS c47_z FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) a UNION ALL SELECT stack(24, 'cod_recebedor', 'string', c48_n, c48_v, c48_z, 'nm_recebedor', 'string', c49_n, c49_v, c49_z, 'desc_endereco_entrega', 'string', c50_n, c50_v, c50_z, 'cod_cep_entrega', 'string', c51_n, c51_v, c51_z, 'desc_bairro_entrega', 'string', c52_n, c52_v, c52_z, 'nm_cidade_entrega', 'string', c53_n, c53_v, c53_z, 'cod_incoterms_1', 'string', c54_n, c54_v, c54_z, 'cod_incoterms_2', 'string', c55_n, c55_v, c55_z, 'dt_data_emissao_mercadoria', 'date', c56_n, c56_v, c56_z, 'nm_cidade', 'string', c57_n, c57_v, c57_z, 'cod_transportador', 'string', c58_n, c58_v, c58_z, 'cod_placa_cavalo', 'string', c59_n, c59_v, c59_z, 'cod_transportador_auxiliar', 'string', c60_n, c60_v, c60_z, 'dt_data_entrada_auxiliar', 'date', c61_n, c61_v, c61_z, 'nm_motorista', 'string', c62_n, c62_v, c62_z, 'cod_cpf_motorista', 'string', c63_n, c63_v, c63_z, 'sg_unidade_peso_transporte', 'string', c64_n, c64_v, c64_z, 'nm_nome', 'string', c65_n, c65_v, c65_z, 'cod_empresa_auxiliar', 'string', c66_n, c66_v, c66_z, 'cod_centro_destino', 'string', c67_n, c67_v, c67_z, 'cod_area_carregamento', 'string', c68_n, c68_v, c68_z, 'cod_ponto_carregamento', 'string', c69_n, c69_v, c69_z, 'dt_data', 'date', c70_n, c70_v, c70_z, 'hr_hora', 'string', c71_n, c71_v, c71_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`cod_recebedor` IS NULL) AS c48_n, COUNT_IF(`cod_recebedor` IS NOT NULL AND LOWER(TRIM(`cod_recebedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c48_v, COUNT_IF(TRIM(`cod_recebedor`) RLIKE '^0+([.,]0+)?$') AS c48_z, COUNT_IF(`nm_recebedor` IS NULL) AS c49_n, COUNT_IF(`nm_recebedor` IS NOT NULL AND LOWER(TRIM(`nm_recebedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c49_v, COUNT_IF(TRIM(`nm_recebedor`) RLIKE '^0+([.,]0+)?$') AS c49_z, COUNT_IF(`desc_endereco_entrega` IS NULL) AS c50_n, COUNT_IF(`desc_endereco_entrega` IS NOT NULL AND LOWER(TRIM(`desc_endereco_entrega`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c50_v, COUNT_IF(TRIM(`desc_endereco_entrega`) RLIKE '^0+([.,]0+)?$') AS c50_z, COUNT_IF(`cod_cep_entrega` IS NULL) AS c51_n, COUNT_IF(`cod_cep_entrega` IS NOT NULL AND LOWER(TRIM(`cod_cep_entrega`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c51_v, COUNT_IF(TRIM(`cod_cep_entrega`) RLIKE '^0+([.,]0+)?$') AS c51_z, COUNT_IF(`desc_bairro_entrega` IS NULL) AS c52_n, COUNT_IF(`desc_bairro_entrega` IS NOT NULL AND LOWER(TRIM(`desc_bairro_entrega`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c52_v, COUNT_IF(TRIM(`desc_bairro_entrega`) RLIKE '^0+([.,]0+)?$') AS c52_z, COUNT_IF(`nm_cidade_entrega` IS NULL) AS c53_n, COUNT_IF(`nm_cidade_entrega` IS NOT NULL AND LOWER(TRIM(`nm_cidade_entrega`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c53_v, COUNT_IF(TRIM(`nm_cidade_entrega`) RLIKE '^0+([.,]0+)?$') AS c53_z, COUNT_IF(`cod_incoterms_1` IS NULL) AS c54_n, COUNT_IF(`cod_incoterms_1` IS NOT NULL AND LOWER(TRIM(`cod_incoterms_1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c54_v, COUNT_IF(TRIM(`cod_incoterms_1`) RLIKE '^0+([.,]0+)?$') AS c54_z, COUNT_IF(`cod_incoterms_2` IS NULL) AS c55_n, COUNT_IF(`cod_incoterms_2` IS NOT NULL AND LOWER(TRIM(`cod_incoterms_2`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c55_v, COUNT_IF(TRIM(`cod_incoterms_2`) RLIKE '^0+([.,]0+)?$') AS c55_z, COUNT_IF(`dt_data_emissao_mercadoria` IS NULL) AS c56_n, CAST(0 AS BIGINT) AS c56_v, CAST(0 AS BIGINT) AS c56_z, COUNT_IF(`nm_cidade` IS NULL) AS c57_n, COUNT_IF(`nm_cidade` IS NOT NULL AND LOWER(TRIM(`nm_cidade`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c57_v, COUNT_IF(TRIM(`nm_cidade`) RLIKE '^0+([.,]0+)?$') AS c57_z, COUNT_IF(`cod_transportador` IS NULL) AS c58_n, COUNT_IF(`cod_transportador` IS NOT NULL AND LOWER(TRIM(`cod_transportador`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c58_v, COUNT_IF(TRIM(`cod_transportador`) RLIKE '^0+([.,]0+)?$') AS c58_z, COUNT_IF(`cod_placa_cavalo` IS NULL) AS c59_n, COUNT_IF(`cod_placa_cavalo` IS NOT NULL AND LOWER(TRIM(`cod_placa_cavalo`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c59_v, COUNT_IF(TRIM(`cod_placa_cavalo`) RLIKE '^0+([.,]0+)?$') AS c59_z, COUNT_IF(`cod_transportador_auxiliar` IS NULL) AS c60_n, COUNT_IF(`cod_transportador_auxiliar` IS NOT NULL AND LOWER(TRIM(`cod_transportador_auxiliar`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c60_v, COUNT_IF(TRIM(`cod_transportador_auxiliar`) RLIKE '^0+([.,]0+)?$') AS c60_z, COUNT_IF(`dt_data_entrada_auxiliar` IS NULL) AS c61_n, CAST(0 AS BIGINT) AS c61_v, CAST(0 AS BIGINT) AS c61_z, COUNT_IF(`nm_motorista` IS NULL) AS c62_n, COUNT_IF(`nm_motorista` IS NOT NULL AND LOWER(TRIM(`nm_motorista`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c62_v, COUNT_IF(TRIM(`nm_motorista`) RLIKE '^0+([.,]0+)?$') AS c62_z, COUNT_IF(`cod_cpf_motorista` IS NULL) AS c63_n, COUNT_IF(`cod_cpf_motorista` IS NOT NULL AND LOWER(TRIM(`cod_cpf_motorista`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c63_v, COUNT_IF(TRIM(`cod_cpf_motorista`) RLIKE '^0+([.,]0+)?$') AS c63_z, COUNT_IF(`sg_unidade_peso_transporte` IS NULL) AS c64_n, COUNT_IF(`sg_unidade_peso_transporte` IS NOT NULL AND LOWER(TRIM(`sg_unidade_peso_transporte`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c64_v, COUNT_IF(TRIM(`sg_unidade_peso_transporte`) RLIKE '^0+([.,]0+)?$') AS c64_z, COUNT_IF(`nm_nome` IS NULL) AS c65_n, COUNT_IF(`nm_nome` IS NOT NULL AND LOWER(TRIM(`nm_nome`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c65_v, COUNT_IF(TRIM(`nm_nome`) RLIKE '^0+([.,]0+)?$') AS c65_z, COUNT_IF(`cod_empresa_auxiliar` IS NULL) AS c66_n, COUNT_IF(`cod_empresa_auxiliar` IS NOT NULL AND LOWER(TRIM(`cod_empresa_auxiliar`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c66_v, COUNT_IF(TRIM(`cod_empresa_auxiliar`) RLIKE '^0+([.,]0+)?$') AS c66_z, COUNT_IF(`cod_centro_destino` IS NULL) AS c67_n, COUNT_IF(`cod_centro_destino` IS NOT NULL AND LOWER(TRIM(`cod_centro_destino`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c67_v, COUNT_IF(TRIM(`cod_centro_destino`) RLIKE '^0+([.,]0+)?$') AS c67_z, COUNT_IF(`cod_area_carregamento` IS NULL) AS c68_n, COUNT_IF(`cod_area_carregamento` IS NOT NULL AND LOWER(TRIM(`cod_area_carregamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c68_v, COUNT_IF(TRIM(`cod_area_carregamento`) RLIKE '^0+([.,]0+)?$') AS c68_z, COUNT_IF(`cod_ponto_carregamento` IS NULL) AS c69_n, COUNT_IF(`cod_ponto_carregamento` IS NOT NULL AND LOWER(TRIM(`cod_ponto_carregamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c69_v, COUNT_IF(TRIM(`cod_ponto_carregamento`) RLIKE '^0+([.,]0+)?$') AS c69_z, COUNT_IF(`dt_data` IS NULL) AS c70_n, CAST(0 AS BIGINT) AS c70_v, CAST(0 AS BIGINT) AS c70_z, COUNT_IF(`hr_hora` IS NULL) AS c71_n, COUNT_IF(`hr_hora` IS NOT NULL AND LOWER(TRIM(`hr_hora`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c71_v, COUNT_IF(TRIM(`hr_hora`) RLIKE '^0+([.,]0+)?$') AS c71_z FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) a UNION ALL SELECT stack(11, 'dt_data_inicio_carregamento', 'date', c72_n, c72_v, c72_z, 'hr_hora_inicio_carregamento', 'string', c73_n, c73_v, c73_z, 'dt_data_fim_carregamento', 'date', c74_n, c74_v, c74_z, 'hr_hora_fim_carregamento', 'string', c75_n, c75_v, c75_z, 'cod_usuario_responsavel', 'string', c76_n, c76_v, c76_z, 'sg_unidade_medida_base', 'string', c77_n, c77_v, c77_z, 'sg_moeda', 'string', c78_n, c78_v, c78_z, 'dt_data_sistema', 'date', c79_n, c79_v, c79_z, 'dateingest', 'date', c80_n, c80_v, c80_z, 'yearingest', 'int', c81_n, c81_v, c81_z, 'monthingest', 'int', c82_n, c82_v, c82_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`dt_data_inicio_carregamento` IS NULL) AS c72_n, CAST(0 AS BIGINT) AS c72_v, CAST(0 AS BIGINT) AS c72_z, COUNT_IF(`hr_hora_inicio_carregamento` IS NULL) AS c73_n, COUNT_IF(`hr_hora_inicio_carregamento` IS NOT NULL AND LOWER(TRIM(`hr_hora_inicio_carregamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c73_v, COUNT_IF(TRIM(`hr_hora_inicio_carregamento`) RLIKE '^0+([.,]0+)?$') AS c73_z, COUNT_IF(`dt_data_fim_carregamento` IS NULL) AS c74_n, CAST(0 AS BIGINT) AS c74_v, CAST(0 AS BIGINT) AS c74_z, COUNT_IF(`hr_hora_fim_carregamento` IS NULL) AS c75_n, COUNT_IF(`hr_hora_fim_carregamento` IS NOT NULL AND LOWER(TRIM(`hr_hora_fim_carregamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c75_v, COUNT_IF(TRIM(`hr_hora_fim_carregamento`) RLIKE '^0+([.,]0+)?$') AS c75_z, COUNT_IF(`cod_usuario_responsavel` IS NULL) AS c76_n, COUNT_IF(`cod_usuario_responsavel` IS NOT NULL AND LOWER(TRIM(`cod_usuario_responsavel`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c76_v, COUNT_IF(TRIM(`cod_usuario_responsavel`) RLIKE '^0+([.,]0+)?$') AS c76_z, COUNT_IF(`sg_unidade_medida_base` IS NULL) AS c77_n, COUNT_IF(`sg_unidade_medida_base` IS NOT NULL AND LOWER(TRIM(`sg_unidade_medida_base`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c77_v, COUNT_IF(TRIM(`sg_unidade_medida_base`) RLIKE '^0+([.,]0+)?$') AS c77_z, COUNT_IF(`sg_moeda` IS NULL) AS c78_n, COUNT_IF(`sg_moeda` IS NOT NULL AND LOWER(TRIM(`sg_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c78_v, COUNT_IF(TRIM(`sg_moeda`) RLIKE '^0+([.,]0+)?$') AS c78_z, COUNT_IF(`dt_data_sistema` IS NULL) AS c79_n, CAST(0 AS BIGINT) AS c79_v, CAST(0 AS BIGINT) AS c79_z, COUNT_IF(`dateingest` IS NULL) AS c80_n, CAST(0 AS BIGINT) AS c80_v, CAST(0 AS BIGINT) AS c80_z, COUNT_IF(`yearingest` IS NULL) AS c81_n, CAST(0 AS BIGINT) AS c81_v, COUNT_IF(`yearingest` = 0) AS c81_z, COUNT_IF(`monthingest` IS NULL) AS c82_n, CAST(0 AS BIGINT) AS c82_v, COUNT_IF(`monthingest` = 0) AS c82_z FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) a), t AS (SELECT COUNT(*) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) SELECT p.coluna, p.tipo, t.total, p.nulos, p.vazios, p.zeros, t.total - p.nulos - p.vazios AS preenchidos_nao_vazios, CASE WHEN t.total = 0 THEN 'SEM LINHAS' WHEN p.nulos = t.total THEN '100% NULO - CONFERIR COMENTARIO' WHEN p.nulos + p.vazios = t.total THEN '100% SEM VALOR UTIL - CONFERIR' ELSE 'COM DADOS' END AS veredito FROM perf p CROSS JOIN t ORDER BY p.coluna;


## 6.1. Colunas sem fonte

Não há coluna explicitamente classificada como sem fonte no DESCRIBE; não conclui que todas são preenchidas.


In [0]:
SELECT 'Nenhuma coluna explicitamente classificada como sem fonte no DESCRIBE' AS observacao;


## 7. Cardinalidade

Identifica atributos constantes, categorias e possíveis identificadores sem expor valores pessoais.


In [0]:
WITH a AS (SELECT COUNT(*) AS total, COUNT(DISTINCT `num_documento`) AS d0, COUNT(DISTINCT `num_material`) AS d1, COUNT(DISTINCT `cod_empresa`) AS d2, COUNT(DISTINCT `cod_local_negocio`) AS d3, COUNT(DISTINCT `cod_codigo_cliente`) AS d4, COUNT(DISTINCT `tp_categoria_nota_fiscal`) AS d5, COUNT(DISTINCT `ind_estornada`) AS d6, COUNT(DISTINCT `ind_impressa`) AS d7, COUNT(DISTINCT `cod_centro`) AS d8 FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) SELECT coluna, distintos, total, CASE WHEN distintos <= 1 THEN 'CONSTANTE OU VAZIO' WHEN distintos <= 20 THEN 'BAIXA CARDINALIDADE' WHEN distintos >= total * 0.9 THEN 'CANDIDATO A IDENTIFICADOR' ELSE 'INTERMEDIARIA' END AS perfil FROM (SELECT total, stack(9, 'num_documento', d0, 'num_material', d1, 'cod_empresa', d2, 'cod_local_negocio', d3, 'cod_codigo_cliente', d4, 'tp_categoria_nota_fiscal', d5, 'ind_estornada', d6, 'ind_impressa', d7, 'cod_centro', d8) AS (coluna, distintos) FROM a) s ORDER BY distintos DESC;


## 8. Domínio

Mostra até oito valores por campo categórico não pessoal; nomes, identificadores pessoais e endereços ficam fora.


In [0]:
WITH d AS (SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `cod_empresa` UNION ALL SELECT 'cod_local_negocio' AS coluna, CAST(`cod_local_negocio` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `cod_local_negocio` UNION ALL SELECT 'cod_codigo_cliente' AS coluna, CAST(`cod_codigo_cliente` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `cod_codigo_cliente` UNION ALL SELECT 'tp_categoria_nota_fiscal' AS coluna, CAST(`tp_categoria_nota_fiscal` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `tp_categoria_nota_fiscal` UNION ALL SELECT 'ind_estornada' AS coluna, CAST(`ind_estornada` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `ind_estornada` UNION ALL SELECT 'ind_impressa' AS coluna, CAST(`ind_impressa` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `ind_impressa` UNION ALL SELECT 'ind_manual' AS coluna, CAST(`ind_manual` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `ind_manual` UNION ALL SELECT 'cod_fatura' AS coluna, CAST(`cod_fatura` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `cod_fatura`), r AS (SELECT coluna, valor, linhas, ROW_NUMBER() OVER (PARTITION BY coluna ORDER BY linhas DESC, valor) AS pos FROM d) SELECT coluna, valor, linhas FROM r WHERE pos <= 8 ORDER BY coluna, pos;


## 9. Booleanas e indicadores

Mede os estados observados, sem converter indicadores STRING em booleanos por suposição.


In [0]:
WITH a AS (SELECT 'tp_categoria_nota_fiscal' AS coluna, CAST(`tp_categoria_nota_fiscal` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `tp_categoria_nota_fiscal` UNION ALL SELECT 'ind_estornada' AS coluna, CAST(`ind_estornada` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `ind_estornada` UNION ALL SELECT 'ind_impressa' AS coluna, CAST(`ind_impressa` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `ind_impressa` UNION ALL SELECT 'ind_manual' AS coluna, CAST(`ind_manual` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `ind_manual`) SELECT coluna, valor, linhas FROM a ORDER BY coluna, linhas DESC;


## 10. Perfil numérico

Perfila quantidades e valores. Campos de códigos numéricos não são tratados como montantes.


In [0]:
SELECT * FROM (SELECT stack(5, 'qt_quantidade', CAST(n0_min AS DOUBLE), CAST(n0_max AS DOUBLE), CAST(n0_avg AS DOUBLE), CAST(n0_med AS DOUBLE), n0_neg, n0_zero, 'vl_frete', CAST(n1_min AS DOUBLE), CAST(n1_max AS DOUBLE), CAST(n1_avg AS DOUBLE), CAST(n1_med AS DOUBLE), n1_neg, n1_zero, 'qt_peso_bruto', CAST(n2_min AS DOUBLE), CAST(n2_max AS DOUBLE), CAST(n2_avg AS DOUBLE), CAST(n2_med AS DOUBLE), n2_neg, n2_zero, 'qt_peso_liquido', CAST(n3_min AS DOUBLE), CAST(n3_max AS DOUBLE), CAST(n3_avg AS DOUBLE), CAST(n3_med AS DOUBLE), n3_neg, n3_zero, 'qt_peso_liquido_nf', CAST(n4_min AS DOUBLE), CAST(n4_max AS DOUBLE), CAST(n4_avg AS DOUBLE), CAST(n4_med AS DOUBLE), n4_neg, n4_zero) AS (coluna, minimo, maximo, media, mediana_aproximada, negativos, zeros) FROM (SELECT MIN(`qt_quantidade`) AS n0_min, MAX(`qt_quantidade`) AS n0_max, AVG(`qt_quantidade`) AS n0_avg, PERCENTILE_APPROX(`qt_quantidade`, 0.5) AS n0_med, COUNT_IF(`qt_quantidade` < 0) AS n0_neg, COUNT_IF(`qt_quantidade` = 0) AS n0_zero, MIN(`vl_frete`) AS n1_min, MAX(`vl_frete`) AS n1_max, AVG(`vl_frete`) AS n1_avg, PERCENTILE_APPROX(`vl_frete`, 0.5) AS n1_med, COUNT_IF(`vl_frete` < 0) AS n1_neg, COUNT_IF(`vl_frete` = 0) AS n1_zero, MIN(`qt_peso_bruto`) AS n2_min, MAX(`qt_peso_bruto`) AS n2_max, AVG(`qt_peso_bruto`) AS n2_avg, PERCENTILE_APPROX(`qt_peso_bruto`, 0.5) AS n2_med, COUNT_IF(`qt_peso_bruto` < 0) AS n2_neg, COUNT_IF(`qt_peso_bruto` = 0) AS n2_zero, MIN(`qt_peso_liquido`) AS n3_min, MAX(`qt_peso_liquido`) AS n3_max, AVG(`qt_peso_liquido`) AS n3_avg, PERCENTILE_APPROX(`qt_peso_liquido`, 0.5) AS n3_med, COUNT_IF(`qt_peso_liquido` < 0) AS n3_neg, COUNT_IF(`qt_peso_liquido` = 0) AS n3_zero, MIN(`qt_peso_liquido_nf`) AS n4_min, MAX(`qt_peso_liquido_nf`) AS n4_max, AVG(`qt_peso_liquido_nf`) AS n4_avg, PERCENTILE_APPROX(`qt_peso_liquido_nf`, 0.5) AS n4_med, COUNT_IF(`qt_peso_liquido_nf` < 0) AS n4_neg, COUNT_IF(`qt_peso_liquido_nf` = 0) AS n4_zero FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) a) n ORDER BY coluna;


## 11. Totais para conciliação

Soma separadamente cada quantidade e valor. Totais DOUBLE são aproximados; comparar moedas e granularidade antes de conciliar.


In [0]:
SELECT coluna, total FROM (SELECT stack(5, 'qt_quantidade', CAST(s0 AS DOUBLE), 'vl_frete', CAST(s1 AS DOUBLE), 'qt_peso_bruto', CAST(s2 AS DOUBLE), 'qt_peso_liquido', CAST(s3 AS DOUBLE), 'qt_peso_liquido_nf', CAST(s4 AS DOUBLE)) AS (coluna, total) FROM (SELECT SUM(`qt_quantidade`) AS s0, SUM(`vl_frete`) AS s1, SUM(`qt_peso_bruto`) AS s2, SUM(`qt_peso_liquido`) AS s3, SUM(`qt_peso_liquido_nf`) AS s4 FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) a) s ORDER BY coluna;


## 12. Datas

Verifica sentinelas e formatos; mínimos de campos STRING são literais, não datas cronológicas.


In [0]:
SELECT * FROM (SELECT 'dt_data_documento' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data_documento` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data_documento` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data_documento`) AS STRING) AS menor_literal, CAST(MAX(`dt_data_documento`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dt_data_entrada_portaria' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data_entrada_portaria` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data_entrada_portaria` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data_entrada_portaria`) AS STRING) AS menor_literal, CAST(MAX(`dt_data_entrada_portaria`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dt_data_chamado' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data_chamado` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data_chamado` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data_chamado`) AS STRING) AS menor_literal, CAST(MAX(`dt_data_chamado`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dt_data_encaminhado' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data_encaminhado` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data_encaminhado` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data_encaminhado`) AS STRING) AS menor_literal, CAST(MAX(`dt_data_encaminhado`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dt_data_faturamento' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data_faturamento` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data_faturamento` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data_faturamento`) AS STRING) AS menor_literal, CAST(MAX(`dt_data_faturamento`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dt_data_saida_portaria' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data_saida_portaria` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data_saida_portaria` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data_saida_portaria`) AS STRING) AS menor_literal, CAST(MAX(`dt_data_saida_portaria`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dt_data_retornado' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data_retornado` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data_retornado` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data_retornado`) AS STRING) AS menor_literal, CAST(MAX(`dt_data_retornado`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dt_data_liberacao_2a_vez' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data_liberacao_2a_vez` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data_liberacao_2a_vez` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data_liberacao_2a_vez`) AS STRING) AS menor_literal, CAST(MAX(`dt_data_liberacao_2a_vez`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dt_data_emissao_mercadoria' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data_emissao_mercadoria` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data_emissao_mercadoria` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data_emissao_mercadoria`) AS STRING) AS menor_literal, CAST(MAX(`dt_data_emissao_mercadoria`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dt_data_entrada_auxiliar' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data_entrada_auxiliar` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data_entrada_auxiliar` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data_entrada_auxiliar`) AS STRING) AS menor_literal, CAST(MAX(`dt_data_entrada_auxiliar`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dt_data' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data`) AS STRING) AS menor_literal, CAST(MAX(`dt_data`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dt_data_inicio_carregamento' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data_inicio_carregamento` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data_inicio_carregamento` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data_inicio_carregamento`) AS STRING) AS menor_literal, CAST(MAX(`dt_data_inicio_carregamento`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dt_data_fim_carregamento' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data_fim_carregamento` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data_fim_carregamento` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data_fim_carregamento`) AS STRING) AS menor_literal, CAST(MAX(`dt_data_fim_carregamento`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dt_data_sistema' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_data_sistema` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_data_sistema` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_data_sistema`) AS STRING) AS menor_literal, CAST(MAX(`dt_data_sistema`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'dateingest' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dateingest` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dateingest` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dateingest`) AS STRING) AS menor_literal, CAST(MAX(`dateingest`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) d ORDER BY coluna;


## 13. Códigos

Detecta zeros à esquerda e possíveis colisões ao normalizar identificadores não pessoais. Não normalizar sem avaliar a semântica.


In [0]:
SELECT *, codigos_numericos_distintos - codigos_normalizados_distintos AS colisoes_potenciais FROM (SELECT 'num_documento' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_documento`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_documento`))) AS tamanho_max, COUNT_IF(TRIM(`num_documento`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_documento`) RLIKE '^[0-9]+$' THEN TRIM(`num_documento`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_documento`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_documento`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'cod_empresa' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_empresa`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_empresa`))) AS tamanho_max, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_empresa`) RLIKE '^[0-9]+$' THEN TRIM(`cod_empresa`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_empresa`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_empresa`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'cod_local_negocio' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_local_negocio`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_local_negocio`))) AS tamanho_max, COUNT_IF(TRIM(`cod_local_negocio`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_local_negocio`) RLIKE '^[0-9]+$' THEN TRIM(`cod_local_negocio`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_local_negocio`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_local_negocio`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'cod_codigo_cliente' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_codigo_cliente`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_codigo_cliente`))) AS tamanho_max, COUNT_IF(TRIM(`cod_codigo_cliente`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_codigo_cliente`) RLIKE '^[0-9]+$' THEN TRIM(`cod_codigo_cliente`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_codigo_cliente`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_codigo_cliente`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'num_nota_fiscal' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_nota_fiscal`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_nota_fiscal`))) AS tamanho_max, COUNT_IF(TRIM(`num_nota_fiscal`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_nota_fiscal`) RLIKE '^[0-9]+$' THEN TRIM(`num_nota_fiscal`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_nota_fiscal`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_nota_fiscal`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'num_einvoice' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_einvoice`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_einvoice`))) AS tamanho_max, COUNT_IF(TRIM(`num_einvoice`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_einvoice`) RLIKE '^[0-9]+$' THEN TRIM(`num_einvoice`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_einvoice`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_einvoice`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'num_material' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_material`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_material`))) AS tamanho_max, COUNT_IF(TRIM(`num_material`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_material`) RLIKE '^[0-9]+$' THEN TRIM(`num_material`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_material`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_material`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'num_documento_transporte' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_documento_transporte`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_documento_transporte`))) AS tamanho_max, COUNT_IF(TRIM(`num_documento_transporte`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_documento_transporte`) RLIKE '^[0-9]+$' THEN TRIM(`num_documento_transporte`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_documento_transporte`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_documento_transporte`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'cod_fatura' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_fatura`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_fatura`))) AS tamanho_max, COUNT_IF(TRIM(`cod_fatura`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_fatura`) RLIKE '^[0-9]+$' THEN TRIM(`cod_fatura`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_fatura`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_fatura`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` UNION ALL SELECT 'cod_direcao' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_direcao`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_direcao`))) AS tamanho_max, COUNT_IF(TRIM(`cod_direcao`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_direcao`) RLIKE '^[0-9]+$' THEN TRIM(`cod_direcao`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_direcao`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_direcao`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) c ORDER BY coluna;


## 14. Checksum

Conta linhas 100% idênticas por hash da linha inteira; não retorna conteúdo nem hashes de dados pessoais.


In [0]:
WITH h AS (SELECT md5(to_json(struct(t.`num_documento`, t.`cod_empresa`, t.`cod_local_negocio`, t.`cod_codigo_cliente`, t.`num_nota_fiscal`, t.`num_einvoice`, t.`cod_chave_acesso`, t.`dt_data_documento`, t.`tp_categoria_nota_fiscal`, t.`ind_estornada`, t.`num_material`, t.`cod_container`, t.`num_documento_transporte`, t.`nm_cliente`, t.`desc_endereco_chegada`, t.`ind_impressa`, t.`ind_manual`, t.`cod_fatura`, t.`cod_direcao`, t.`desc_material`, t.`qt_quantidade`, t.`vl_frete`, t.`qt_peso_bruto`, t.`qt_peso_liquido`, t.`qt_peso_liquido_nf`, t.`sg_unidade_peso`, t.`cod_roteiro`, t.`desc_roteiro`, t.`cod_centro`, t.`num_processo`, t.`cod_evento_processo`, t.`cod_portaria_entrada`, t.`cod_portaria_saida`, t.`dt_data_entrada_portaria`, t.`hr_hora_entrada_portaria`, t.`dt_data_chamado`, t.`hr_hora_chamado`, t.`dt_data_encaminhado`, t.`hr_hora_encaminhado`, t.`dt_data_faturamento`, t.`hr_hora_faturamento`, t.`dt_data_saida_portaria`, t.`hr_hora_saida_portaria`, t.`dt_data_retornado`, t.`hr_hora_retornado`, t.`dt_data_liberacao_2a_vez`, t.`hr_hora_liberacao_2a_vez`, t.`num_remessa`, t.`cod_recebedor`, t.`nm_recebedor`, t.`desc_endereco_entrega`, t.`cod_cep_entrega`, t.`desc_bairro_entrega`, t.`nm_cidade_entrega`, t.`cod_incoterms_1`, t.`cod_incoterms_2`, t.`dt_data_emissao_mercadoria`, t.`nm_cidade`, t.`cod_transportador`, t.`cod_placa_cavalo`, t.`cod_transportador_auxiliar`, t.`dt_data_entrada_auxiliar`, t.`nm_motorista`, t.`cod_cpf_motorista`, t.`sg_unidade_peso_transporte`, t.`nm_nome`, t.`cod_empresa_auxiliar`, t.`cod_centro_destino`, t.`cod_area_carregamento`, t.`cod_ponto_carregamento`, t.`dt_data`, t.`hr_hora`, t.`dt_data_inicio_carregamento`, t.`hr_hora_inicio_carregamento`, t.`dt_data_fim_carregamento`, t.`hr_hora_fim_carregamento`, t.`cod_usuario_responsavel`, t.`sg_unidade_medida_base`, t.`sg_moeda`, t.`dt_data_sistema`, t.`dateingest`, t.`yearingest`, t.`monthingest`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` t), a AS (SELECT hash_linha, COUNT(*) AS linhas FROM h GROUP BY hash_linha) SELECT COALESCE(SUM(linhas),0) AS linhas, COUNT(*) AS linhas_distintas, COALESCE(SUM(linhas - 1),0) AS repeticoes_identicas FROM a;


## 15. Amostra

Mostra 20 registros com colunas não pessoais; campos potencialmente pessoais são omitidos ou hash de 8 caracteres.


In [0]:
SELECT `num_documento`, `cod_empresa`, `cod_local_negocio`, `cod_codigo_cliente`, `num_nota_fiscal`, `num_einvoice`, `dt_data_documento`, `tp_categoria_nota_fiscal`, `ind_estornada`, `num_material`, `num_documento_transporte`, `ind_impressa`, `ind_manual`, `cod_fatura`, `cod_direcao`, `desc_material`, SUBSTRING(MD5(CAST(`cod_chave_acesso` AS STRING)),1,8) AS `cod_chave_acesso_hash`, SUBSTRING(MD5(CAST(`cod_container` AS STRING)),1,8) AS `cod_container_hash`, SUBSTRING(MD5(CAST(`nm_cliente` AS STRING)),1,8) AS `nm_cliente_hash`, SUBSTRING(MD5(CAST(`desc_endereco_chegada` AS STRING)),1,8) AS `desc_endereco_chegada_hash`, SUBSTRING(MD5(CAST(`cod_recebedor` AS STRING)),1,8) AS `cod_recebedor_hash` FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` LIMIT 20;


## 16. Distribuição interna

Compara o mix de categorias e centro, quando houver, sem expor pessoas.


In [0]:
SELECT `cod_centro`, `cod_empresa`, `cod_local_negocio`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `cod_centro`, `cod_empresa`, `cod_local_negocio` ORDER BY linhas DESC LIMIT 100;


## 17. Freshness

Mede datas de carga e partições; não confundir carimbo de ingestão com evento de negócio.


In [0]:
SELECT COUNT(*) AS linhas, MIN(`dateingest`) AS `min_dateingest`, MAX(`dateingest`) AS `max_dateingest`, COUNT(DISTINCT `dateingest`) AS `distintos_dateingest`, MIN(`yearingest`) AS `min_yearingest`, MAX(`yearingest`) AS `max_yearingest`, COUNT(DISTINCT `yearingest`) AS `distintos_yearingest`, MIN(`monthingest`) AS `min_monthingest`, MAX(`monthingest`) AS `max_monthingest`, COUNT(DISTINCT `monthingest`) AS `distintos_monthingest` FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`;


## 18. Análises específicas da transação

Testes derivados dos comentários do DESCRIBE; as hipóteses dependem dos resultados medidos.


### 18.1. Cobertura da expedição e do transporte


In [0]:
SELECT `cod_centro`, COUNT(*) AS linhas, COUNT_IF(`num_einvoice` IS NOT NULL AND TRIM(`num_einvoice`) <> '') AS einvoice_preenchido, COUNT_IF(`cod_chave_acesso` IS NOT NULL AND TRIM(`cod_chave_acesso`) <> '') AS chave_acesso_preenchida, COUNT_IF(`dt_data_entrada_portaria` IS NOT NULL) AS entrada_preenchida, COUNT_IF(`dt_data_saida_portaria` IS NOT NULL) AS saida_preenchida, COUNT_IF(`dt_data_entrada_portaria` IS NOT NULL AND `dt_data_saida_portaria` IS NOT NULL AND `dt_data_saida_portaria` < `dt_data_entrada_portaria`) AS saida_antes_entrada FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `cod_centro` ORDER BY linhas DESC;


## 19. Amostra ampliada

Amostra de até 500 registros sem campos pessoais literais; não equivale à extração completa do cenário.


In [0]:
SELECT `num_documento`, `cod_empresa`, `cod_local_negocio`, `cod_codigo_cliente`, `num_nota_fiscal`, `num_einvoice`, `dt_data_documento`, `tp_categoria_nota_fiscal`, `ind_estornada`, `num_material`, `num_documento_transporte`, `ind_impressa`, `ind_manual`, `cod_fatura`, `cod_direcao`, `desc_material`, SUBSTRING(MD5(CAST(`cod_chave_acesso` AS STRING)),1,8) AS `cod_chave_acesso_hash`, SUBSTRING(MD5(CAST(`cod_container` AS STRING)),1,8) AS `cod_container_hash`, SUBSTRING(MD5(CAST(`nm_cliente` AS STRING)),1,8) AS `nm_cliente_hash`, SUBSTRING(MD5(CAST(`desc_endereco_chegada` AS STRING)),1,8) AS `desc_endereco_chegada_hash`, SUBSTRING(MD5(CAST(`cod_recebedor` AS STRING)),1,8) AS `cod_recebedor_hash` FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` LIMIT 500;


## 20. Resumo

Copie este bloco junto com as seções 3, 5, 6 e 18 para validar chave real e filtros antes dos cenários.


In [0]:
WITH d AS (SELECT `num_documento`, `num_material`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao` GROUP BY `num_documento`, `num_material`), v AS (SELECT COUNT(*) AS linhas, COUNT(DISTINCT `cod_centro`) AS centros_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao`) SELECT 'ZVCSD5077_001' AS transacao, 'vw_ds_com_zvcsd5077_001_nota_fiscal_expedicao' AS objeto, v.linhas, (SELECT COUNT(*) FROM d) AS chaves_candidatas_distintas, (SELECT COUNT(*) FROM d WHERE linhas > 1) AS chaves_candidatas_repetidas, v.centros_distintos, 'Chave candidata: num_documento + num_material' AS nota_chave FROM v;


## Checklist para comparação com SAP

- [ ] Confirmar comentário do objeto e filtros aplicados na tela SAP.
- [ ] Registrar instante de execução e mesmo recorte temporal da extração SAP.
- [ ] Validar chave real, duplicatas idênticas e versões distintas.
- [ ] Conferir centros 4128 e 4014 somente quando há campo de centro; caso contrário, definir correspondência de negócio antes de comparar.
- [ ] Verificar nulos sem declaração de sem fonte; não classificar lacuna documentada como defeito.
- [ ] Normalizar códigos com zeros à esquerda e datas; validar separador decimal, moeda e tolerância de 0,005 para DOUBLE.
- [ ] Não extrair nem divulgar valores pessoais literais.
- [ ] Executar cenários somente após receber os resultados deste notebook.
